# 1.4. Pruebas de hipótesis

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency, ttest_ind

ALFA = 0.05
ruta_relativa = Path("data/raw/weatherAUS.csv")
ruta_csv = next(
    (carpeta / ruta_relativa for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ruta_relativa).is_file()),
    None,
)
if ruta_csv is None:
    raise FileNotFoundError(ruta_relativa)

datos = pd.read_csv(ruta_csv)
assert datos.shape == (145_460, 23)
print(f"Filas: {len(datos):,} | Columnas: {datos.shape[1]}".replace(",", "."))

categorias = ["No", "Yes"]

def formato_p(valor_p):
    if valor_p == 0:
        return "p < 1e-300 (por debajo de la precisión numérica)"
    return f"p = {valor_p:.3g}"

def decision(valor_p):
    return "Se rechaza H₀" if valor_p < ALFA else "No se rechaza H₀"

Filas: 145.460 | Columnas: 23


## t de Welch bilateral

- **H₀**. La media de `Humidity3pm` es igual para `RainTomorrow = Yes` y `RainTomorrow = No`.
- **H₁**. Las medias de `Humidity3pm` difieren entre ambos grupos.

Se usa la t de Welch porque permite varianzas distintas entre los grupos. La prueba es bilateral y su nivel de significancia es α = 0,05.

In [2]:
welch = datos.loc[
    datos["RainTomorrow"].isin(categorias) & datos["Humidity3pm"].notna(),
    ["RainTomorrow", "Humidity3pm"],
]
grupos = welch.groupby("RainTomorrow")["Humidity3pm"]
resumen = grupos.agg(
    N="size", Media="mean", **{"Desviación estándar": "std"}
).reindex(["Yes", "No"])
print(f"Filas válidas: {len(welch)} | Filas excluidas: {len(datos) - len(welch)}")
display(resumen.round(3))

si = grupos.get_group("Yes")
no = grupos.get_group("No")
diferencia = si.mean() - no.mean()
prueba_t = ttest_ind(si, no, equal_var=False, alternative="two-sided")

print(f"Diferencia de medias (Yes menos No): {diferencia:.3f} puntos porcentuales")
print(f"t = {prueba_t.statistic:.3f} | {formato_p(prueba_t.pvalue)}")
print(
    f"{decision(prueba_t.pvalue)} con α = {ALFA:.2f}. "
    f"La humedad media observada fue de {si.mean():.2f} % para Yes "
    f"y {no.mean():.2f} % para No, una diferencia de {diferencia:.2f} puntos porcentuales."
)

Filas válidas: 138583 | Filas excluidas: 6877

,N,Media,Desviación estándar
RainTomorrow,,,
Yes,30913,68.800,19.037
No,107670,46.511,18.489


Diferencia de medias (Yes menos No): 22.289 puntos porcentuales
t = 182.608 | p < 1e-300 (por debajo de la precisión numérica)
Se rechaza H₀ con α = 0.05. La humedad media observada fue de 68.80 % para Yes y 46.51 % para No, una diferencia de 22.29 puntos porcentuales.


## Chi-cuadrado de independencia

- **H₀**. `RainToday` y `RainTomorrow` son independientes.
- **H₁**. Existe una asociación entre `RainToday` y `RainTomorrow`.

Se aplica la prueba chi-cuadrado de Pearson sin corrección de continuidad, con α = 0,05. Las frecuencias esperadas se calculan a partir de los totales marginales bajo H₀.

In [3]:
lluvia = datos.loc[
    datos["RainToday"].isin(categorias) & datos["RainTomorrow"].isin(categorias),
    ["RainToday", "RainTomorrow"],
]
observadas = pd.crosstab(lluvia["RainToday"], lluvia["RainTomorrow"]).reindex(
    index=categorias, columns=categorias
)
prueba_chi = chi2_contingency(observadas, correction=False)
esperadas = pd.DataFrame(
    prueba_chi.expected_freq, index=observadas.index, columns=observadas.columns
)

print(f"Filas válidas: {len(lluvia)} | Filas excluidas: {len(datos) - len(lluvia)}")
print("Frecuencias observadas")
display(observadas)
print("Frecuencias esperadas bajo independencia")
display(esperadas.round(2))
print(
    f"χ² = {prueba_chi.statistic:.3f} | Grados de libertad = {prueba_chi.dof} "
    f"| {formato_p(prueba_chi.pvalue)}"
)

porcentajes = observadas["Yes"].div(observadas.sum(axis=1)).mul(100)
print(
    f"{decision(prueba_chi.pvalue)} con α = {ALFA:.2f}. "
    f"Llovió al día siguiente en el {porcentajes['Yes']:.2f} % de las filas con lluvia hoy "
    f"y en el {porcentajes['No']:.2f} % de las filas sin lluvia hoy. "
    f"La diferencia observada fue de {porcentajes['Yes'] - porcentajes['No']:.2f} "
    "puntos porcentuales."
)

Filas válidas: 140787 | Filas excluidas: 4673
Frecuencias observadas


RainTomorrow,No,Yes
RainToday,,
No,92728,16604
Yes,16858,14597


Frecuencias esperadas bajo independencia


RainTomorrow,No,Yes
RainToday,,
No,85102.01,24229.99
Yes,24483.99,6971.01


χ² = 13801.289 | Grados de libertad = 1 | p < 1e-300 (por debajo de la precisión numérica)
Se rechaza H₀ con α = 0.05. Llovió al día siguiente en el 46.41 % de las filas con lluvia hoy y en el 15.19 % de las filas sin lluvia hoy. La diferencia observada fue de 31.22 puntos porcentuales.


## Alcance de los resultados

Las observaciones de una misma estación o de días consecutivos pueden estar relacionadas. Ambas pruebas calculan los valores p bajo un supuesto de independencia entre observaciones. Si existe dependencia, esos valores p pueden quedar mal calibrados. Los contrastes describen asociaciones en las observaciones analizadas y el diseño observacional limita cualquier conclusión causal.